# 一个Simulation，按需逐单元

先配置和检查，再选择静态或动态；同一个sim保留当前A/B，没有必需的stages清单。这是3×3×1仅各向异性的有限示例，不是材料或网格收敛证明。科学单元会求场；只运行导入和模型配置不等于已执行科学单元。安装、实际执行与新内核的资格须分别查看验收记录。

选择新安装候选对应内核，不要热重载旧模拟；不在Notebook安装包、不依赖.command。[后处理教程](notebook_skyrmion_postprocess.ipynb) · [公式](../docs/MATHEMATICS.md) · [全部参数](../docs/API_REFERENCE.md)。


In [ ]:
from pathlib import Path
import fimrun.micromagnetics as mm
from fimrun.notebook import NotebookSimulation


## 1. 模型、单位与初态

$$\mathbf M_i=M_{s,i}\mathbf m_i,\quad|\mathbf m_i|=1,\quad\overline{\mathbf M}=(M_{s,A}\mathbf m_A+M_{s,B}\mathbf m_B)/2.$$

长度m、时间s、Ms及H用A/m；B_ext用T。A/B不是上下层；Antiparallel明确产生B=−A，材料仍可不同。仅energy中列出的项参与。


In [ ]:
grid = mm.Grid(cells=(3, 3, 1), cell_size=(4e-9, 4e-9, 3e-9))  # m
sub_A = mm.Sublattice(Ms=3e5, alpha=0.01, gamma_H=2e5)
sub_B = mm.Sublattice(Ms=6e5, alpha=0.01, gamma_H=2e5)
model = mm.FiM(mesh=grid, material=mm.FiMMaterial(A=sub_A, B=sub_B),
            energy=(mm.FiMAnisotropy(K1_A=1e4, K1_B=2e4),))
initial = mm.Antiparallel(mm.Uniform((0., 0., 1.)))


## 2. 创建并检查

只需模型、初态和输出目录；创建即显示初态并复用同一viewer，不求场。动态dt尚未配置。重新构造闲置sim会回初态，续算请保留同一sim。旧结果不会自动覆盖，科学操作遇同名需明确确认。请在新的输出目录执行教程。


In [ ]:
output_dir = (Path.cwd() / 'results' / 'new_notebook_demo').resolve()
sim = NotebookSimulation(model=model, initial=initial, output_dir=output_dir)
sim


## 3. 需要静态时执行

$$R_i=\max_j|\mathbf H_{i,j}-(\mathbf m_{i,j}\cdot\mathbf H_{i,j})\mathbf m_{i,j}|,\qquad X_i=R_i/M_{s,i}.$$

residual是R的A/m门，mxh是X的无量纲门，A/B在同一接受态同时满足；consecutive数不同接受态。此轴向均匀小例初态即满足。静态不增加物理时间；save_step是接受迭代间隔，不是秒或迭代上限。未达标的真实案例可能很长，请用Interrupt停止；没有隐藏长程预算。


In [ ]:
sim.relax(name='static', mxh=1e-5, residual=1.0, save_step=1)


## 4. 明确加电流，再动态

电流不是默认值。J单位A/m²，xi及场型比无量纲；d×J决定极化方向。这一步接刚才的A/B，不重新初始化。要无电流则用`sim.set_current(None)`。


In [ ]:
sim.set_current(mm.FiMSOT(J=(1e11, 0., 0.), xi_A=0.1, xi_B=-0.1,
    incident_direction=(0., 0., 1.), field_like_A=0., field_like_B=0.))


## 5. 动态有限示例

$$\dot{\mathbf m}_i=-\frac{\gamma_{H,i}}{1+\alpha_i^2}[\mathbf m_i\times\mathbf H_i+\alpha_i\mathbf m_i\times(\mathbf m_i\times\mathbf H_i)].$$

gamma_H采用H约定，m/(A s)；本例duration=dt=1fs，仅一接受步。真实问题自行选择时间与dt，不把这个样例当采样充分性。save_interval用秒并与dt对齐。


In [ ]:
sim.run(name='drive', dt=1e-15, duration=1e-15, save_interval=1e-15)


## 6. 保存、恢复与观察

保存使用新名字，原结果不覆盖；普通state与strict checkpoint等级不同，不能跨实现假装strict恢复。磁态导出为独立A/B方向；只看结果不必运行后处理。


In [ ]:
sim.save('final.state')
sim.export_ovf('current')


### 同一个viewer与恢复

创建时已打开viewer，可用`sim.preview()`重新显示同一页面；网页暂停观看不停止计算。3D显示部分网格的清晰箭头，2D可显示完整单层XY热图，显示密度不改变计算网格。

新简洁路径默认全部七种scalar和A/B OVF，不自动结构分析。save_step是静态接受迭代间隔；save_interval是动态秒数并与dt对齐。省略频率沿用该类阶段已配置频率，新会话未配置则仅末态；显式None也是仅末态，不新增自动100帧或额外初态采样。CSV/OVF共享频率，接受态索引去重；能量列见[输出说明](../docs/interface-v01/NOTEBOOK_OUTPUTS.md)。

Restart后在新目录用`NotebookSimulation.load(path, output_dir=..., name='restored')`，恢复磁态、累计时间、电流、dt及输出设置；普通state重建私有历史，strict checkpoint必须同实现。无动态dt的纯静态state恢复后仍无dt。完成后`sim.close()`。


In [ ]:
sim.close()
